# Advanced Balanced Hybrid Persona Pipeline

This notebook operationalizes the balanced hybrid system in a way that is ready for later integration into a contextual engineering pipeline.

It does five things:

1. Loads the uploaded guide, schema, and codebook.
2. Builds an anchor registry from grounded persona cards.
3. Expands coverage with controlled PersonaHub-style generation plans.
4. Validates candidates with schema checks, drift checks, and realism scoring.
5. Emits context-ready packets that can later plug into a Pragmatist-style injection and evaluation workflow.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ARTIFACT_DIR = Path(".").resolve()
MODULE_PATH = ARTIFACT_DIR / "advanced_balanced_hybrid_persona_pipeline.py"
GUIDE_PATH = ARTIFACT_DIR / "hybrid_persona_system_guide.md"
SCHEMA_PATH = ARTIFACT_DIR / "hybrid_persona_schema.json"
CODEBOOK_PATH = ARTIFACT_DIR / "hybrid_persona_label_codebook.csv"
OUTPUT_DIR = ARTIFACT_DIR / "advanced_hybrid_demo_outputs_v5"

print(MODULE_PATH)
print(GUIDE_PATH)
print(SCHEMA_PATH)
print(CODEBOOK_PATH)
print(OUTPUT_DIR)

## Why this notebook is more advanced than a simple anchor-plus-expansion script

The underlying module adds several control layers that make the hybrid system much more robust:

- **Coverage-aware budgeting** so generation is steered toward underrepresented intent and output slices.
- **Anchor-conditioned few-shot selection** using MMR-style diversity rather than random exemplar pickup.
- **Transparent hybrid labeling** that combines codebook cues, tie-breakers, and lexical fallback.
- **Writer-critic style validation** with schema checks, label agreement scoring, drift flags, and realism scoring.
- **Context-ready state objects** with global memory, session memory, and injection-ready renderers.
- **A/B test hooks** for comparing different context selection strategies before you plug this into an agent workflow.


In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("advanced_pipeline", MODULE_PATH)
advanced_pipeline = importlib.util.module_from_spec(spec)
sys.modules["advanced_pipeline"] = advanced_pipeline
spec.loader.exec_module(advanced_pipeline)

advanced_pipeline.PipelineConfig()

## Load the uploaded artifacts

The module uses the uploaded CSV codebook as executable label rules and validates output records against the uploaded JSON schema.


In [ ]:
codebook_df = advanced_pipeline.load_codebook_csv(CODEBOOK_PATH)
schema = advanced_pipeline.load_json_schema(SCHEMA_PATH)
guide_text = GUIDE_PATH.read_text(encoding="utf-8")

display(codebook_df.head())
print("Schema title:", schema.get("title"))
print("Guide preview:", guide_text[:500])

## Initialize the pipeline

The default configuration is already tuned for a balanced pilot:
- 24 generated candidates per anchor
- 30/45/25 intent mix
- 50/30/20 output-type mix
- automatic balancing toward ~12 selected records per anchor


In [ ]:
config = advanced_pipeline.PipelineConfig(
    seed=7,
    default_budget_per_anchor=24,
    approve_threshold=0.60,
    reject_threshold=0.42,
    target_records_per_anchor=12,
)
pipeline = advanced_pipeline.BalancedHybridPipeline(codebook_df, schema, config=config)
config

## Register anchor personas

Swap `demo_anchor_cards()` with your own Script-generated anchor cards when you have them.

Expected columns:
- `anchor_persona_id`
- `macro_persona`
- `anchor_name`
- `job_to_be_done`
- `constraints`
- `success_metric`
- `decision_criteria`
- `vocabulary`
- `evidence_sources`
- `confidence_score`
- `priority_topics`


In [ ]:
anchors_df = advanced_pipeline.demo_anchor_cards()
pipeline.register_anchors(anchors_df)

pd.DataFrame([
    {
        "anchor_persona_id": a.anchor_persona_id,
        "macro_persona": a.macro_persona,
        "anchor_name": a.anchor_name,
        "priority_topics": ", ".join(a.priority_topics),
        "vocabulary": ", ".join(a.vocabulary[:6]),
    }
    for a in pipeline.state.anchor_registry.values()
])

## Ingest observed or Script-grounded records

For a real run, replace `demo_observed_records()` with:
- your GSC export
- Script-generated grounded prompts
- survey-derived rows
- transcript-derived rows

At minimum, the dataframe needs a `text` column.


In [ ]:
observed_df = advanced_pipeline.demo_observed_records()
annotated_observed = pipeline.ingest_observed_records(
    observed_df,
    source_mode="observed",
    assign_anchor=False,
)

annotated_observed[[
    "text",
    "anchor_persona_id",
    "persona_macro",
    "intent_level",
    "topic_product_category",
    "label_confidence",
]].head(10)

## Build the controlled generation plan

This is where the balanced method differs from a simpler workflow:

- It uses the current dataset to allocate a budget to underfilled slices.
- It keeps the anchor persona fixed.
- It alternates between within-anchor and stakeholder-neighbor expansion.
- It selects few-shot exemplars per intent band instead of mixing everything together.


In [ ]:
specs = pipeline.plan_generation()
plan_df = pd.DataFrame([{
    "anchor_persona_id": s.anchor_persona_id,
    "persona_micro": s.persona_micro,
    "topic_product_category": s.topic_product_category,
    "intent_level": s.intent_level,
    "output_type": s.output_type,
    "mode": s.mode,
    "num_exemplars": len(s.exemplars),
} for s in specs])

plan_df.head(12)

## Generate and validate candidates

The default adapter is a deterministic notebook-safe stand-in for real PersonaHub-driven or LLM-driven generation.  
Later you can replace it with a custom adapter that calls an actual model.

Validation includes:
- JSON schema validation
- rule-based relabeling
- persona / intent / topic drift detection
- vocabulary and constraint alignment checks
- realism scoring


In [ ]:
candidates = pipeline.generate_candidates()
summary = pipeline.evaluate(candidates)

summary

In [ ]:
candidate_df = pd.DataFrame([c.to_dict() for c in candidates])
candidate_df[[
    "anchor_persona_id",
    "persona_macro",
    "intent_level",
    "topic_product_category",
    "output_type",
    "text",
    "realism_score",
    "review_status",
    "drift_flags",
]].sort_values(["review_status", "realism_score"], ascending=[True, False]).head(20)

## Rebalance and consolidate

The rebalance step tries to keep the final set closer to the target intent and output-type mix while capping over-repeated micro-personas.  
The consolidation step converts approved records into a global memory store and keeps the remaining candidate set in session memory, which makes later context injection easier.


In [ ]:
balanced_records = pipeline.rebalance(candidates)
consolidation_report = pipeline.consolidate(balanced_records)

print("Balanced record count:", len(balanced_records))
print("Balanced status counts:", pd.Series([r.review_status for r in balanced_records]).value_counts().to_dict())
consolidation_report

## Build a context packet for downstream context engineering

This is the forward-looking bridge to the follow-up integration work.

The packet includes:
- YAML frontmatter
- selected global memories
- selected session memories
- a context-policy block
- audit metadata so you can detect context errors early


In [ ]:
query = "best zero trust access tools for remote employees with sso"
packet = pipeline.build_context_packet(query, strategy="anchor_priority")

print(packet.frontmatter)
print()
print(packet.global_memories_md[:1000])
print()
print(packet.audit)

## A/B compare context selection strategies

This mirrors the strategy-comparison pattern in Pragmatist-style pipelines.


In [ ]:
ab_df = advanced_pipeline.compare_context_strategies(
    pipeline.state,
    pipeline.labeler,
    queries=[
        "best zero trust access tools for remote employees with sso",
        "best note taking app for college with offline sync",
        "edr pricing for 100 endpoints",
    ],
    config=pipeline.config,
)

ab_df

## Export reusable artifacts

These are the files you can reuse in the next integration step.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

advanced_pipeline.export_records_jsonl(candidates, OUTPUT_DIR / "generated_candidates.jsonl")
advanced_pipeline.export_records_jsonl(balanced_records, OUTPUT_DIR / "balanced_selected_records.jsonl")
advanced_pipeline.export_context_packet(packet, OUTPUT_DIR / "context_packet.yaml")
ab_df.to_csv(OUTPUT_DIR / "context_strategy_comparison.csv", index=False)
(OUTPUT_DIR / "generation_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(OUTPUT_DIR / "consolidation_report.json").write_text(json.dumps(consolidation_report, indent=2), encoding="utf-8")

sorted(str(p) for p in OUTPUT_DIR.iterdir())

## Swapping in a real model later

The module already contains `build_llm_generation_prompt(spec, anchor)`.

In the follow-up integration step, the simplest upgrade path is:

1. Keep the current planning, balancing, validation, and export logic.
2. Replace `HeuristicPersonaHubAdapter` with an adapter that calls your model.
3. Route approved records into the same state object used by your broader context pipeline.
4. Use `compare_context_strategies()` and packet audits to test context-injection choices before deployment.
